# Machine Learning Basics: Finding the Best Function
### Model · Loss · Optimization

At its core, **machine learning is automatic function finding**. Instead of hand-writing the rule that maps an input `x` to an output `y`, we let an algorithm *search* for a good function for us.

This search always follows the same **three steps**:

1. **Model** — define a *set of candidate functions* $\mathcal{H}$ (e.g. all lines $y = wx + b$). This is the search space.
2. **Loss** — define a number $L(f)$ that measures how *bad* a candidate function $f$ is on our data. Smaller is better.
3. **Optimization** — search the candidate set for the best function:
$$f^* = \arg\min_{f \in \mathcal{H}} L(f).$$

### Roadmap of this notebook

- **Framework** (C03–C06): build a *Model* and a *Loss* on one simple linear-regression example, and feel the loss with sliders.
- **How we optimize** (C07–C09): implement **gradient descent** and watch the *learning rate* control convergence, divergence, and oscillation.
- **Why optimization alone is not enough** (C10–C13): a low *training* loss can still generalize poorly — see **overfitting** and tame it with **regularization**.

Everything runs on small synthetic data, top-to-bottom, no GPU required. Let's begin.

In [ ]:
# C02 — Setup: imports, reproducibility, and plotting helpers
# Install ipywidgets if it is missing (harmless on Colab where it is preinstalled).
# !pip install -q ipywidgets

import numpy as np
import matplotlib.pyplot as plt

# Inline plotting (guarded so this cell also runs outside IPython).
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Interactive widgets (guarded: fall back to static plots if unavailable).
try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not found -> interactive cells will render a static fallback.")
    print("Tip: install it with  !pip install ipywidgets")

# Reproducibility: one shared seeded RNG used by every data-generating cell.
SEED = 42
rng = np.random.default_rng(SEED)

# Consistent figure styling.
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 110


def style_axes(ax, title=None, xlabel='x', ylabel='y'):
    """Apply consistent labels, optional title, and a light grid to an Axes."""
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    ax.grid(alpha=0.3)
    return ax


# Quick self-checks.
assert 'np' in dir() and 'plt' in dir()
assert isinstance(rng, np.random.Generator)

print(f"Environment ready | numpy {np.__version__} | matplotlib {plt.matplotlib.__version__} | seed={SEED}")
print(f"ipywidgets available: {WIDGETS_AVAILABLE}")

## Step 1 — Model: the candidate function set $\mathcal{H}$

The **model** is the *set of functions we are willing to consider*. We call it $\mathcal{H}$ (the *hypothesis set*). Learning means picking the single best function out of this set.

The simplest interesting model is **linear regression**:

$$f_{w,b}(x) = w\,x + b.$$

Here every choice of the parameters $(w, b)$ gives a *different candidate function* — a different straight line. So $\mathcal{H}$ is the infinite family of all lines, and our job is to find the $(w, b)$ whose line best matches the data.

**Running example.** Throughout the Model / Loss / Optimization part of this notebook we use one fixed synthetic dataset: points generated from a true line $y = w_{\text{true}}\,x + b_{\text{true}}$ plus a little noise. Because we know the true parameters, we can check whether our search actually recovers them.

In [ ]:
# C04 — Synthetic linear dataset (created once, reused by C06, C08, C09)
true_w, true_b = 2.0, -1.0
N = 40
noise_std = 1.0

# Sorted x for clean left-to-right plotting; draw from the shared seeded rng.
x_train = np.sort(rng.uniform(-3, 3, size=N))
y_train = true_w * x_train + true_b + rng.normal(0, noise_std, size=N)

assert x_train.shape == y_train.shape == (N,)
assert np.all(np.diff(x_train) >= 0)  # x is sorted

# One example candidate line (a guess, NOT the truth) to make 'the function set' concrete.
w_guess, b_guess = 1.0, 0.0
xs = np.linspace(x_train.min(), x_train.max(), 100)

fig, ax = plt.subplots()
ax.scatter(x_train, y_train, color='steelblue', label='training data')
ax.plot(xs, w_guess * xs + b_guess, color='crimson', lw=2,
        label=f'one candidate: y={w_guess}x+{b_guess}')
style_axes(ax, title=f'Synthetic data from true line  y = {true_w}x + ({true_b})')
ax.legend()
plt.show()

print(f"Generated N={N} points. True parameters: w={true_w}, b={true_b}")

## Step 2 — Loss: how bad is a candidate function?

Given a candidate $f_{w,b}$, the **loss** $L(w, b)$ aggregates the gap between the function's predictions and the correct labels. The slides describe it as a *sum of distances* between each prediction and its target.

We use the **Mean Squared Error (MSE)**:

$$L(w, b) = \frac{1}{N}\sum_{i=1}^{N}\bigl(f_{w,b}(x_i) - y_i\bigr)^2 = \frac{1}{N}\sum_{i=1}^{N}\bigl((w x_i + b) - y_i\bigr)^2.$$

Key properties:

- $L \ge 0$ always, and $L = 0$ only if the line passes through every point exactly.
- Squaring makes large errors hurt disproportionately and makes $L$ smooth (nicely differentiable), which the optimizer in Step 3 will exploit.

A smaller loss means a better candidate. In the next cell you can move sliders for $w$ and $b$ and *watch the loss change* — that is the quantity optimization will try to minimize.

In [ ]:
# C06 — MSE loss + interactive (w, b) explorer
def loss(w, b):
    """Mean squared error of the line y = w*x + b on the training data."""
    preds = w * x_train + b
    return float(np.mean((preds - y_train) ** 2))


# Fixed view limits (computed once) so the plot does not jump as sliders move.
_pad = 0.1 * (y_train.max() - y_train.min())
_ylim = (y_train.min() - _pad - 2, y_train.max() + _pad + 2)
_xs = np.linspace(x_train.min(), x_train.max(), 100)


def plot_line(w, b):
    fig, ax = plt.subplots()
    ax.scatter(x_train, y_train, color='steelblue', label='data')
    ax.plot(_xs, w * _xs + b, color='crimson', lw=2, label='candidate line')
    ax.set_ylim(_ylim)
    style_axes(ax, title=f'Loss  L(w={w:.2f}, b={b:.2f}) = {loss(w, b):.3f}')
    ax.legend(loc='upper left')
    plt.show()


# Sanity checks.
assert loss(true_w, true_b) >= 0.0
assert loss(true_w, true_b) < loss(0.0, 0.0)  # the truth fits better than the flat zero line

if WIDGETS_AVAILABLE:
    interact(plot_line,
             w=FloatSlider(min=-1, max=5, step=0.1, value=1.0, description='w'),
             b=FloatSlider(min=-5, max=5, step=0.1, value=0.0, description='b'))
else:
    print("Static fallback (widgets unavailable): showing the true-parameter line.")
    plot_line(true_w, true_b)

## Step 3 — Optimization: searching for $f^*$

We now want the *best* candidate:

$$f^* = \arg\min_{w, b} L(w, b).$$

We could never try every $(w, b)$ by hand. Instead we use **gradient descent**, an iterative rule that repeatedly nudges the parameters *downhill* on the loss surface.

The gradient $\nabla L = \left(\dfrac{\partial L}{\partial w}, \dfrac{\partial L}{\partial b}\right)$ points in the direction of *steepest increase*, so we step in the **opposite** direction:

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}.$$

For MSE the gradients have a clean closed form (with $e_i = (w x_i + b) - y_i$):

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i e_i\,x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{N}\sum_i e_i.$$

The step size $\eta$ is the **learning rate** — a *hyperparameter* you choose. Too small and learning crawls; too large and the updates overshoot and may diverge. We explore exactly this trade-off next.

In [ ]:
# C08 — Gradient descent for linear-regression MSE (reused by C09)
def gradient_descent(lr, n_steps, w0=0.0, b0=0.0):
    """Run gradient descent on the MSE loss.

    Returns a trajectory dict of float arrays {'w', 'b', 'loss'} including the
    initial point. Stops early (returning the partial path) if the loss blows up.
    """
    n_steps = max(1, int(n_steps))
    w, b = float(w0), float(b0)
    ws, bs, ls = [w], [b], [loss(w, b)]
    for _ in range(n_steps):
        preds = w * x_train + b
        err = preds - y_train
        grad_w = (2.0 / N) * np.sum(err * x_train)
        grad_b = (2.0 / N) * np.sum(err)
        w -= lr * grad_w
        b -= lr * grad_b
        cur = loss(w, b)
        ws.append(w); bs.append(b); ls.append(cur)
        if not np.isfinite(cur):  # diverged -> keep partial trajectory for C09
            break
    return {'w': np.array(ws, dtype=float),
            'b': np.array(bs, dtype=float),
            'loss': np.array(ls, dtype=float)}


# Example run + comparison against the known truth.
traj = gradient_descent(lr=0.05, n_steps=50)
assert len(traj['w']) == len(traj['loss'])
print(f"Final GD estimate : w={traj['w'][-1]:.3f}, b={traj['b'][-1]:.3f}, loss={traj['loss'][-1]:.3f}")
print(f"True parameters   : w={true_w:.3f}, b={true_b:.3f}, loss={loss(true_w, true_b):.3f}")

# Validation: a longer run should converge near the truth.
_t = gradient_descent(0.05, 200)
assert _t['loss'][-1] < _t['loss'][0]
assert abs(_t['w'][-1] - true_w) < 0.5 and abs(_t['b'][-1] - true_b) < 0.5

In [ ]:
# C09 — Interactive gradient-descent visualizer
# Precompute the loss surface once (kept out of the callback for responsiveness).
W = np.linspace(-1, 5, 80)
B = np.linspace(-5, 5, 80)
WW, BB = np.meshgrid(W, B)
ZZ = np.empty_like(WW)
for i in range(WW.shape[0]):
    for j in range(WW.shape[1]):
        ZZ[i, j] = loss(WW[i, j], BB[i, j])
assert ZZ.shape == WW.shape == BB.shape


def show_descent(lr, n_steps):
    t = gradient_descent(lr, n_steps)
    ws, bs, ls = t['w'], t['b'], t['loss']
    finite = np.isfinite(ws) & np.isfinite(bs) & np.isfinite(ls)
    diverged = not np.all(finite)

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8))

    # Left: loss-surface contour + descent path.
    cf = axL.contourf(WW, BB, ZZ, levels=30, cmap='viridis')
    axL.contour(WW, BB, ZZ, levels=12, colors='white', linewidths=0.4, alpha=0.5)
    fig.colorbar(cf, ax=axL, label='loss')
    axL.plot(ws[finite], bs[finite], color='white', lw=1.5, marker='o',
             markersize=3, label='descent path')
    axL.scatter([true_w], [true_b], color='red', marker='*', s=200,
                edgecolor='white', zorder=5, label='truth')
    style_axes(axL, title='Loss surface + path', xlabel='w', ylabel='b')
    axL.legend(loc='upper right')

    # Right: loss vs iteration.
    lf = ls[finite]
    if lf.size > 0 and np.all(lf > 0):
        axR.semilogy(np.arange(lf.size), lf, color='crimson', marker='.')
        axR.set_ylabel('loss (log scale)')
    else:
        axR.plot(np.arange(lf.size), lf, color='crimson', marker='.')
        axR.set_ylabel('loss')
    axR.set_xlabel('iteration')
    title = 'diverged (try a smaller learning rate)' if diverged else f'final loss = {ls[-1]:.3f}'
    axR.set_title(title)
    axR.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


if WIDGETS_AVAILABLE:
    interact(show_descent,
             lr=FloatLogSlider(base=10, min=-3, max=0.5, step=0.1, value=0.05,
                               description='learning rate'),
             n_steps=IntSlider(min=5, max=300, step=5, value=50, description='steps'))
else:
    print("Static fallback (widgets unavailable): convergent run lr=0.05, n_steps=50.")
    show_descent(0.05, 50)

## From fitting to generalizing: the overfitting trap

Gradient descent can drive the **training** loss very low. But the real goal of machine learning is to do well on *new, unseen* data — to **generalize**.

A model can achieve near-zero training loss and still be useless on new inputs. This is **overfitting**: the model effectively *memorizes* the training points (including their noise) instead of learning the underlying pattern. It is like memorizing the answers to a practice exam rather than understanding the material.

To detect this, we split our data:

- **Training set** — used to fit the model.
- **Test set** — held out, used only to *evaluate* generalization.

Watch for the tell-tale gap: as a model becomes more flexible, **training error keeps dropping while test error eventually rises**. The next demo makes this gap visible by fitting polynomials of increasing degree to a nonlinear dataset.

In [ ]:
# C11 — Nonlinear dataset + train/test split (created once, reused by C12, C13)
def g(x):
    """True underlying nonlinear function."""
    return np.sin(1.5 * x)


M = 30
x_all = np.sort(rng.uniform(-3, 3, size=M))
y_all = g(x_all) + rng.normal(0, 0.25, size=M)

# Shuffled split: ~60% train, rest test.
idx = rng.permutation(M)
cut = int(round(0.6 * M))
train_idx, test_idx = idx[:cut], idx[cut:]

# Re-sort each split by x for clean curve overlays.
train_idx = train_idx[np.argsort(x_all[train_idx])]
test_idx = test_idx[np.argsort(x_all[test_idx])]
x_train2, y_train2 = x_all[train_idx], y_all[train_idx]
x_test2,  y_test2  = x_all[test_idx],  y_all[test_idx]

assert len(x_train2) + len(x_test2) == M
assert len(x_train2) > 0 and len(x_test2) > 0
assert len(set(train_idx.tolist()) & set(test_idx.tolist())) == 0  # disjoint

_grid = np.linspace(x_all.min(), x_all.max(), 200)
fig, ax = plt.subplots()
ax.plot(_grid, g(_grid), color='gray', ls='--', lw=1.5, label='true function g(x)=sin(1.5x)')
ax.scatter(x_train2, y_train2, color='steelblue', label=f'train (n={len(x_train2)})')
ax.scatter(x_test2, y_test2, color='darkorange', marker='^', label=f'test (n={len(x_test2)})')
style_axes(ax, title='Nonlinear data: train vs test split')
ax.legend()
plt.show()

In [ ]:
# C12 — Interactive polynomial overfitting demo
import warnings


def mse(y_true, y_pred):
    """Mean squared error helper (reused by C13)."""
    return float(np.mean((y_true - y_pred) ** 2))


# np.RankWarning moved to np.exceptions.RankWarning in NumPy 2.0; handle both.
_RankWarning = getattr(np, 'RankWarning', None) or np.exceptions.RankWarning

# Bounded view so exploding high-degree curves stay readable.
_yspan = max(y_train2.max(), y_test2.max()) - min(y_train2.min(), y_test2.min())
_ylim2 = (min(y_train2.min(), y_test2.min()) - 0.5 * _yspan - 1,
          max(y_train2.max(), y_test2.max()) + 0.5 * _yspan + 1)
_pgrid = np.linspace(x_all.min(), x_all.max(), 300)


def plot_poly(degree):
    degree = int(degree)
    degree = min(degree, len(x_train2) - 1)  # cannot exceed n_train - 1
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', _RankWarning)
        try:
            coeffs = np.polyfit(x_train2, y_train2, degree)
        except np.linalg.LinAlgError:
            coeffs = np.polyfit(x_train2, y_train2, min(degree, 3))
    tr = mse(y_train2, np.polyval(coeffs, x_train2))
    te = mse(y_test2, np.polyval(coeffs, x_test2))

    fig, ax = plt.subplots()
    ax.plot(_pgrid, np.polyval(coeffs, _pgrid), color='crimson', lw=2, label='fitted polynomial')
    ax.scatter(x_train2, y_train2, color='steelblue', label='train')
    ax.scatter(x_test2, y_test2, color='darkorange', marker='^', label='test')
    ax.set_ylim(_ylim2)
    style_axes(ax, title=f'degree={degree}   train MSE={tr:.3f}   test MSE={te:.3f}')
    ax.legend(loc='upper left')
    plt.show()


# Sanity: higher degree never increases training error.
_tr1 = mse(y_train2, np.polyval(np.polyfit(x_train2, y_train2, 1), x_train2))
_tr9 = mse(y_train2, np.polyval(np.polyfit(x_train2, y_train2, 9), x_train2))
assert _tr9 <= _tr1 + 1e-9

if WIDGETS_AVAILABLE:
    interact(plot_poly, degree=IntSlider(min=1, max=15, step=1, value=1, description='degree'))
else:
    print("Static fallback (widgets unavailable): showing degree=9 (overfit).")
    plot_poly(9)

In [ ]:
# C13 — Regularization (ridge) remedy at a fixed high degree
DEG = 12


def ridge_fit(X, y, lam):
    """Closed-form ridge regression. The intercept column is NOT penalized."""
    p = X.shape[1]
    R = np.eye(p)
    R[-1, -1] = 0.0  # np.vander's last column is the constant (intercept) term
    A = X.T @ X + lam * R
    b = X.T @ y
    try:
        theta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        theta, *_ = np.linalg.lstsq(A, b, rcond=None)
    return theta


_X_train = np.vander(x_train2, DEG + 1)
_X_test = np.vander(x_test2, DEG + 1)
_X_grid = np.vander(_pgrid, DEG + 1)


def plot_ridge(log_lambda):
    lam = 10.0 ** log_lambda
    theta = ridge_fit(_X_train, y_train2, lam)
    tr = mse(y_train2, _X_train @ theta)
    te = mse(y_test2, _X_test @ theta)

    fig, ax = plt.subplots()
    ax.plot(_pgrid, _X_grid @ theta, color='crimson', lw=2, label=f'ridge fit (deg={DEG})')
    ax.scatter(x_train2, y_train2, color='steelblue', label='train')
    ax.scatter(x_test2, y_test2, color='darkorange', marker='^', label='test')
    ax.set_ylim(_ylim2)
    style_axes(ax, title=f'lambda={lam:.1e}   train MSE={tr:.3f}   test MSE={te:.3f}')
    ax.legend(loc='upper left')
    plt.show()


# Sanity: a larger penalty shrinks the coefficient norm.
_t_small = ridge_fit(_X_train, y_train2, 1e-6)
_t_big = ridge_fit(_X_train, y_train2, 1e3)
assert np.linalg.norm(_t_big) < np.linalg.norm(_t_small)

if WIDGETS_AVAILABLE:
    interact(plot_ridge,
             log_lambda=FloatSlider(min=-6, max=3, step=0.5, value=-3,
                                    description='log10(lambda)'))
else:
    print("Static fallback (widgets unavailable): showing log10(lambda)=-3.")
    plot_ridge(-3)

## Recap & where to go next

We framed machine learning as **automatic function finding** in three steps, and lived through each one:

1. **Model** — we chose linear regression $y = wx + b$, a *set* of candidate functions (C03–C04).
2. **Loss** — we measured each candidate with MSE and *felt* the loss landscape with sliders (C05–C06).
3. **Optimization** — we implemented **gradient descent** and saw the **learning rate** decide between slow convergence, clean convergence, and divergence/oscillation (C07–C09).

Then we confronted the catch: low training loss is not the goal.

- **Overfitting** — high-degree polynomials drove *training* error to ~0 while *test* error climbed (C10–C12).
- **Regularization** — an L2 (ridge) penalty shrank the wild coefficients, smoothed the fit, and narrowed the train/test gap (C13).

### Try extending this

- Swap in other **optimizers** (momentum, Adam) and compare convergence paths on the same loss surface.
- Add **more features** (multivariate regression) or basis functions.
- Move to **classification** with a different loss (cross-entropy) and a sigmoid/softmax model.
- Use **cross-validation** to *choose* the polynomial degree or $\lambda$ automatically instead of by slider.

Same three steps every time: pick a Model, define a Loss, Optimize — and always check that you *generalize*.